# Nexar Collision Prediction — Train/Validation Split and Clip Sampling

**Computer Vision Project**

## Goal

This notebook prepares the dataset for machine-learning experiments.

The main tasks are:

- Create a video-level train/validation split
- Preserve the class distribution
- Define the prediction time for each example
- Define the time-to-event (TTE) conditions
- Convert timestamps into video frame numbers
- Extract temporal clips without using future information
- Define an equivalent sampling strategy for negative videos
- Test the sampling procedure on several videos

The dataset contains complete dashcam videos, but our models will
operate on temporal clips sampled from those videos.

The most important constraint is:

> A model must only receive frames that would have been available at
> the prediction time.

Therefore, for a positive video with an annotated event at time
`t_event`, a TTE of `1.0` second means that the final frame available
to the model is approximately:

`t_prediction = t_event - 1.0`

No frames after this prediction time may be included.

## 1. Imports

We use:

- `pandas` for the metadata
- `numpy` for numerical operations
- `matplotlib` for visualisation
- `cv2` (OpenCV) for reading videos
- `Path` for file paths
- `train_test_split` for creating the train/validation split

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

from pathlib import Path
from sklearn.model_selection import train_test_split


## 2. Define dataset paths

The dataset is stored locally.

We will continue using the same directory structure as the previous
notebooks.

In [4]:
DATA_DIR = Path("./data/raw")

TRAIN_DIR = DATA_DIR / "train"

train_csv = DATA_DIR / "train.csv"

print("Dataset directory:")
print(DATA_DIR.resolve())

print("\nTraining video directory:")
print(TRAIN_DIR.resolve())

Dataset directory:
/Users/rameez/Documents/Wits Stuff/Honours/Computer Vision/nexar-collision-prediction/data/raw

Training video directory:
/Users/rameez/Documents/Wits Stuff/Honours/Computer Vision/nexar-collision-prediction/data/raw/train


## 3. Load the metadata

We load `train.csv`.

Each row represents one training video.

The important columns are:

- `id` — video identifier
- `time_of_event` — annotated event time for positive examples
- `time_of_alert` — annotated alert time for positive examples
- `target` — class label

For negative examples, the event and alert timestamps are missing.

In [5]:
df = pd.read_csv(train_csv)

print("Number of videos:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Number of videos: 1500
Columns: ['id', 'time_of_event', 'time_of_alert', 'target']


,id,time_of_event,time_of_alert,target
0,1924,NaN,NaN,0
1,822,19.5,18.633,1
2,1429,NaN,NaN,0
3,208,19.8,19.233,1
4,1904,NaN,NaN,0


## 4. Create a video-level train/validation split

We split by **video**, not by extracted clips.

This is extremely important.

Suppose one video produces 20 clips.

If 15 clips go into training and 5 clips go into validation, the model
would effectively see parts of the same video during both training and
validation.

That would create data leakage.

Instead:

1. First split the videos.
2. Then generate clips from the videos inside each split.

This ensures that validation videos are completely unseen during
training.

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

print("Training videos:", len(train_df))
print("Validation videos:", len(val_df))

random_state=42 #split is reproducible.

stratify=df["target"] #so that the positive/negative ratio is approximately preserved.


## 5. Check the class distribution

The original dataset is balanced.

We should verify that the train and validation sets remain balanced
after splitting.

In [ ]:
print("Original class distribution:")
print(df["target"].value_counts())

print("\nTraining class distribution:")
print(train_df["target"].value_counts())

print("\nValidation class distribution:")
print(val_df["target"].value_counts())

In [ ]:
print("Training proportions:")
print(train_df["target"].value_counts(normalize=True))

print("\nValidation proportions:")
print(val_df["target"].value_counts(normalize=True))

You should see approximately 50% / 50% in both sets.

Don't worry if the numbers aren't perfectly identical because the split contains a finite number of videos.

## 6. Verify that there is no video overlap

The training and validation sets must contain completely different
video IDs.

We explicitly check this rather than assuming the split worked.

In [ ]:
train_ids = set(train_df["id"])
val_ids = set(val_df["id"])

overlap = train_ids.intersection(val_ids)

print("Number of overlapping videos:", len(overlap))

if len(overlap) == 0:
    print("No video leakage detected.")
else:
    print("WARNING: overlapping video IDs detected.")

We want:

Number of overlapping videos: 0

## 7. Define the TTE conditions

Our main experiment will evaluate prediction at several distances
before the annotated event.

TTE means:

**Time To Event**

For example:

- TTE = 0.5 s → prediction occurs 0.5 seconds before the event
- TTE = 1.0 s → prediction occurs 1.0 second before the event
- TTE = 1.5 s → prediction occurs 1.5 seconds before the event

For a positive video:

`t_prediction = t_event - TTE`

These values will later allow us to measure how early the model can
anticipate a collision/near-miss.

In [ ]:
TTE_VALUES = [0.5, 1.0, 1.5]

print("TTE values:", TTE_VALUES)

8. Define the temporal observation window

Now we need to decide how much video the model sees before the prediction time.

For example, if we use a 2-second observation window:

<---------------- 2 seconds ---------------->

frames used
|--------------------------------------------|
                                             ^
                                      prediction time

                                             |
                                             |
                                           event

For a positive video:

event = 20.0 s
TTE = 1.0 s

prediction time = 19.0 s

With a 2-second observation window:

17.0 s -------------------------- 19.0 s
                                prediction
                                      |
                                      |
                                   20.0 s
                                    event

This means the model sees the 2 seconds immediately preceding the
prediction point, but nothing after it.

We will start with a 2-second observation window.

In [ ]:
CLIP_DURATION = 2.0

print("Clip duration:", CLIP_DURATION, "seconds")

## 9. Convert time into frame numbers

Videos are stored as sequences of frames.

For a video with FPS `f`:

`frame_number ≈ time_seconds × FPS`

For example, if a video has 30 FPS:

- 10 seconds → approximately frame 300
- 15 seconds → approximately frame 450

We will use the video's actual FPS rather than assuming a fixed FPS.

In [ ]:
def time_to_frame(time_seconds, fps):
    frame_number = int(time_seconds * fps)
    return frame_number

## 10. Test the time-to-frame conversion

We can test the function using a hypothetical 30 FPS video.

In [ ]:
fps_example = 30

for time_seconds in [0, 1, 2, 5, 10]:
    frame_number = time_to_frame(time_seconds, fps_example)
    
    print(
        f"{time_seconds} seconds -> "
        f"frame {frame_number}"
    )

Expected:

0 seconds -> frame 0
1 seconds -> frame 30
2 seconds -> frame 60
5 seconds -> frame 150
10 seconds -> frame 300

## 11. Calculate the prediction time

For a positive example:

`prediction_time = time_of_event - TTE`

The event itself is therefore never part of the input clip.

For example:

```text
event = 20.0 s
TTE = 1.5 s

prediction time = 18.5 s
```

The model can only see frames up to approximately 18.5 seconds.

In [ ]:
sample_positive = train_df[train_df["target"] == 1].iloc[0]

event_time = sample_positive["time_of_event"]

print("Video ID:", sample_positive["id"])
print("Event time:", event_time)

for tte in TTE_VALUES:
    prediction_time = event_time - tte
    
    print(
        f"TTE = {tte:.1f}s -> "
        f"prediction time = {prediction_time:.3f}s"
    )

## 12. Calculate the complete temporal clip

For each TTE we calculate:

`prediction_time = event_time - TTE`

Then:

`clip_start = prediction_time - CLIP_DURATION`

and:

`clip_end = prediction_time`

Therefore the clip contains only frames between:

`clip_start <= frame_time <= clip_end`

No future frames are used.

In [ ]:
for tte in TTE_VALUES:
    prediction_time = event_time - tte
    clip_start = prediction_time - CLIP_DURATION
    clip_end = prediction_time
    
    print(f"\nTTE = {tte:.1f}s")
    print("Clip start:", clip_start)
    print("Prediction time:", clip_end)

## 13. Create a function to calculate clip boundaries

We now turn the previous calculation into a reusable function.

The function does not read the video yet.

It only calculates the correct time interval.

In [ ]:
def get_clip_times(event_time, tte, clip_duration):
    prediction_time = event_time - tte
    clip_start = prediction_time - clip_duration
    clip_end = prediction_time
    
    return clip_start, clip_end

In [ ]:
for tte in TTE_VALUES:
    clip_start, clip_end = get_clip_times(
        event_time,
        tte,
        CLIP_DURATION
    )
    
    print(
        f"TTE = {tte:.1f}s -> "
        f"{clip_start:.3f}s to {clip_end:.3f}s"
    )

## 14. Read frames from a temporal clip

Now we move from timestamps to actual video frames.

The function below:

1. Opens the video.
2. Reads the FPS.
3. Converts the start/end times to frame numbers.
4. Reads the frames.
5. Converts OpenCV's BGR representation to RGB.
6. Returns the frames.

We will initially sample a small number of frames from the clip rather
than loading every frame.

In [ ]:
def read_clip(video_path, start_time, end_time, num_frames=16):
    cap = cv2.VideoCapture(str(video_path))
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    start_frame = max(0, int(start_time * fps))
    end_frame = min(
        frame_count - 1,
        int(end_time * fps)
    )
    
    frame_numbers = np.linspace(
        start_frame,
        end_frame,
        num_frames
    ).astype(int)
    
    frames = []
    
    for frame_number in frame_numbers:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        
        success, frame = cap.read()
        
        if success:
            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
            
            frames.append(frame)
    
    cap.release()
    
    return frames, frame_numbers, fps

## 15. Test the clip reader

We will test the function on one positive training video.

The clip will correspond to:

- TTE = 1.0 second
- Observation window = 2 seconds
- 16 sampled frames

In [ ]:
video_id = int(sample_positive["id"])

video_path = TRAIN_DIR / f"{video_id:05d}.mp4"

clip_start, clip_end = get_clip_times(
    sample_positive["time_of_event"],
    1.0,
    CLIP_DURATION
)

frames, frame_numbers, fps = read_clip(
    video_path,
    clip_start,
    clip_end,
    num_frames=16
)

print("Video:", video_id)
print("FPS:", fps)
print("Clip start:", clip_start)
print("Clip end:", clip_end)
print("Number of frames read:", len(frames))
print("Frame numbers:", frame_numbers)

## 16. Check the sampled frame times

We can convert the sampled frame numbers back into approximate times.

This allows us to verify that every frame occurs before the prediction
time.

In [ ]:
frame_times = frame_numbers / fps

for frame_number, frame_time in zip(frame_numbers, frame_times):
    print(
        f"Frame {frame_number} -> "
        f"{frame_time:.3f}s"
    )

In [ ]:
print("Prediction time:", clip_end)
print("Latest sampled frame time:", frame_times[-1])

if frame_times[-1] <= clip_end:
    print("All sampled frames occur at or before prediction time.")
else:
    print("WARNING: a sampled frame occurs after prediction time.")

This is one of the most important checks in the notebook.

## 17. Visualise the sampled clip

Instead of displaying all 16 frames separately, we display them as a
grid.

This allows us to visually confirm that the frames progress through
time.

In [ ]:
fig, axes = plt.subplots(
    4,
    4,
    figsize=(16, 12)
)

for ax, frame, frame_number, frame_time in zip(
    axes.flatten(),
    frames,
    frame_numbers,
    frame_times
):
    ax.imshow(frame)
    ax.axis("off")
    
    ax.set_title(
        f"Frame {frame_number}\n"
        f"{frame_time:.2f}s"
    )

plt.tight_layout()
plt.show()

## 18. Important leakage check

For a positive video, the model must not receive:

- the event frame
- frames after the event
- frames between the prediction point and the event

For example, with:

`TTE = 1.0`

the model must stop approximately one second before the event.

The interval is therefore:

`event - TTE - CLIP_DURATION`
to
`event - TTE`

and **not**:

`event - CLIP_DURATION`
to
`event`

The latter would give the model information from after the intended
prediction point.

## 19. Negative examples

Negative videos do not have an annotated event time.

Therefore, we cannot calculate:

`event - TTE`

for a negative video.

Instead, negative videos need an equivalent temporal sampling strategy.

We will sample a temporal location inside the video and treat that
location as the prediction point.

The negative clip must have the same:

- observation duration
- number of frames
- preprocessing

as the positive clips.

This prevents the classifier from learning differences caused simply
by clip length or sampling procedure.

For the initial implementation we will choose a valid random temporal
location from the negative video.

## 20. Find the duration of a video

We first need a function that returns the video's duration.

In [ ]:
def get_video_properties(video_path):
    cap = cv2.VideoCapture(str(video_path))
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    cap.release()
    
    duration = frame_count / fps
    
    return {
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration": duration
    }

In [ ]:
negative_example = train_df[train_df["target"] == 0].iloc[0]

negative_video_id = int(negative_example["id"])

negative_video_path = (
    TRAIN_DIR / f"{negative_video_id:05d}.mp4"
)

negative_properties = get_video_properties(
    negative_video_path
)

print("Video ID:", negative_video_id)
print("Duration:", negative_properties["duration"])
print("FPS:", negative_properties["fps"])

## 21. Choose a negative prediction time

For a negative video, the prediction time must leave enough room for the
full observation window.

If the clip is 2 seconds long, the prediction point must occur after
the first 2 seconds of the video.

We will initially choose a point in the valid range.

Later, when we perform the full experiment, the exact negative sampling
policy will be fixed and documented so that the experiment is
reproducible.

In [ ]:
negative_duration = negative_properties["duration"]

minimum_prediction_time = CLIP_DURATION
maximum_prediction_time = negative_duration

print("Minimum prediction time:", minimum_prediction_time)
print("Maximum prediction time:", maximum_prediction_time)

## 22. Sample a negative prediction point

We use a fixed random seed so that the selected point is reproducible.

In [ ]:
rng = np.random.default_rng(42)

negative_prediction_time = rng.uniform(
    minimum_prediction_time,
    maximum_prediction_time
)

negative_clip_start = (
    negative_prediction_time - CLIP_DURATION
)

negative_clip_end = negative_prediction_time

print("Negative prediction time:", negative_prediction_time)
print("Negative clip start:", negative_clip_start)
print("Negative clip end:", negative_clip_end)

## 23. Read the negative clip

We use exactly the same `read_clip()` function as for positive
examples.

The only difference is how the prediction time was determined.

In [ ]:
negative_frames, negative_frame_numbers, negative_fps = read_clip(
    negative_video_path,
    negative_clip_start,
    negative_clip_end,
    num_frames=16
)

print("Number of negative frames:", len(negative_frames))
print("FPS:", negative_fps)

In [ ]:
negative_frame_times = (
    negative_frame_numbers / negative_fps
)

print("Negative prediction time:", negative_clip_end)
print(
    "Latest sampled frame:",
    negative_frame_times[-1]
)

if negative_frame_times[-1] <= negative_clip_end:
    print("Negative clip is temporally valid.")
else:
    print("WARNING: invalid negative clip.")

## 24. Visualise the negative clip

In [ ]:
fig, axes = plt.subplots(
    4,
    4,
    figsize=(16, 12)
)

for ax, frame, frame_number, frame_time in zip(
    axes.flatten(),
    negative_frames,
    negative_frame_numbers,
    negative_frame_times
):
    ax.imshow(frame)
    ax.axis("off")
    
    ax.set_title(
        f"Frame {frame_number}\n"
        f"{frame_time:.2f}s"
    )

plt.tight_layout()
plt.show()

## 25. Create a general positive clip sampler

We now combine the previous steps into one function.

The function receives:

- a video path
- event time
- TTE
- clip duration
- number of frames

and returns the sampled frames and their frame numbers.

In [ ]:
def sample_positive_clip(
    video_path,
    event_time,
    tte,
    clip_duration=2.0,
    num_frames=16
):
    clip_start, clip_end = get_clip_times(
        event_time,
        tte,
        clip_duration
    )
    
    frames, frame_numbers, fps = read_clip(
        video_path,
        clip_start,
        clip_end,
        num_frames
    )
    
    return {
        "frames": frames,
        "frame_numbers": frame_numbers,
        "fps": fps,
        "clip_start": clip_start,
        "clip_end": clip_end,
        "tte": tte
    }

## 26. Test the general positive sampler

In [ ]:
positive_clip = sample_positive_clip(
    video_path,
    sample_positive["time_of_event"],
    tte=1.0,
    clip_duration=CLIP_DURATION,
    num_frames=16
)

print("TTE:", positive_clip["tte"])
print("Clip start:", positive_clip["clip_start"])
print("Clip end:", positive_clip["clip_end"])
print("Number of frames:", len(positive_clip["frames"]))

## 27. Check all TTE values

The same positive video should produce valid clips at:

- 0.5 seconds
- 1.0 second
- 1.5 seconds

before the event.

In [ ]:
for tte in TTE_VALUES:
    clip = sample_positive_clip(
        video_path,
        sample_positive["time_of_event"],
        tte=tte,
        clip_duration=CLIP_DURATION,
        num_frames=16
    )
    
    print(
        f"TTE = {tte:.1f}s | "
        f"start = {clip['clip_start']:.3f}s | "
        f"end = {clip['clip_end']:.3f}s | "
        f"frames = {len(clip['frames'])}"
    )

## 28. Check whether all positive training videos support every TTE

A positive example is invalid if:

`event_time - TTE - CLIP_DURATION < 0`

because there would not be enough video before the prediction point to
construct the full observation window.

We count these cases for every TTE.

In [ ]:
for tte in TTE_VALUES:
    clip_start_times = (
        train_df[train_df["target"] == 1]["time_of_event"]
        - tte
        - CLIP_DURATION
    )
    
    invalid = (clip_start_times < 0).sum()
    
    print(
        f"TTE = {tte:.1f}s -> "
        f"{invalid} invalid positive training videos"
    )

This check is important because Notebook 1 showed that event times vary substantially.

We should not assume every positive video has enough pre-event footage.